# Week 1: standard EEG features vs. LaBraM on HBN

This notebook asks the first question of the project: **does a pretrained EEG foundation model (LaBraM) know more about children's resting EEG than the standard features researchers already compute?**

It uses open Healthy Brain Network data (no data agreement needed) and three targets: age, the CBCL attention factor, and the p-factor.

For every child it:

1. downloads the resting-state recording from HBN's public S3 bucket,
2. keeps the eyes-closed blocks, cleans them, and cuts them into 15-second windows on the 19 standard 10-20 channels,
3. computes standard features (band power, relative power, aperiodic slope and offset, alpha peak frequency),
4. runs the windows through pretrained LaBraM and through the same architecture with random weights, saving the average embedding from all 12 layers.

Then it compares the three feature sets with cross-validated ridge regression and with the split-half robustness score from Carson, Carpenter, Dawson & Carlson (2024).

**Before you start:** Runtime > Change runtime type > T4 GPU is nice but not required. Release 1 (about 136 children) takes roughly an hour, mostly downloading.

## 1. Install and set up folders

Saving to Google Drive means you keep the results when Colab disconnects. Skip the Drive cell if you'd rather keep everything temporary.

In [ ]:
!pip install -q braindecode huggingface_hub safetensors mne specparam boto3 scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/autism_eeg_project'

In [ ]:
import os
PROJECT = globals().get('PROJECT', '/content/autism_eeg_project')
DATA_DIR = os.path.join(PROJECT, 'hbn_raw')    # raw files are deleted after processing
OUT_DIR = os.path.join(PROJECT, 'out')
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(PROJECT)
print(PROJECT)

## 2. The pipeline code

This cell saves the pipeline as a file so the next steps can import it. Read through it once: each function matches a step in the project plan's Methods section.

In [ ]:
%%writefile hbn_pipeline.py
"""HBN resting-state pipeline: standard EEG features vs. LaBraM embeddings.

Week 1 of the autism EEG foundation-model project. Everything here runs on
open Healthy Brain Network data (age and CBCL factors), no data agreement needed.
"""
import copy
import os

import boto3
import mne
import numpy as np
import pandas as pd
import torch
from botocore import UNSIGNED
from botocore.config import Config
from scipy.signal import welch
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler, StandardScaler

mne.set_log_level("error")

# ---------------------------------------------------------------- settings
BUCKET = "fcp-indi"
PREFIX = "data/Projects/HBN/BIDS_EEG/cmi_bids_{release}/"

# EGI HydroCel 129 electrodes closest to the standard 10-20 positions
# (the mapping commonly used for HBN / EGI data).
EGI_TO_1020 = {
    "E22": "FP1", "E9": "FP2", "E33": "F7", "E24": "F3", "E11": "FZ",
    "E124": "F4", "E122": "F8", "E45": "T7", "E36": "C3", "Cz": "CZ",
    "E104": "C4", "E108": "T8", "E58": "P7", "E52": "P3", "E62": "PZ",
    "E92": "P4", "E96": "P8", "E70": "O1", "E83": "O2",
}
CH_1020 = list(EGI_TO_1020.values())

SFREQ = 200            # LaBraM expects 200 Hz
WIN_SEC = 15           # the braindecode LaBraM checkpoint takes 3000 samples
WIN = SFREQ * WIN_SEC
SKIP_SEC = 2           # skip 2 s after each "close your eyes" instruction
PTP_REJECT_UV = 500    # drop windows whose peak-to-peak exceeds this
LABRAM_SCALE = 1e4     # volts -> units of 0.1 mV (microvolts / 100), as in LaBraM

BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13),
         "beta": (13, 30), "gamma": (30, 45)}


# ---------------------------------------------------------------- download
def s3_client():
    return boto3.client("s3", config=Config(signature_version=UNSIGNED),
                        region_name="us-east-1")


def get_participants(release, data_dir):
    """Download and return participants.tsv for one HBN release."""
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, f"participants_{release}.tsv")
    if not os.path.exists(path):
        s3_client().download_file(BUCKET, PREFIX.format(release=release)
                                  + "participants.tsv", path)
    return pd.read_csv(path, sep="\t")


def download_resting_state(release, subject, data_dir):
    """Download one child's resting-state .set and events.tsv. Returns paths."""
    s3 = s3_client()
    base = PREFIX.format(release=release) + f"{subject}/eeg/{subject}_task-RestingState_"
    out = {}
    for suffix in ["eeg.set", "events.tsv"]:
        local = os.path.join(data_dir, f"{subject}_RestingState_{suffix}")
        if not os.path.exists(local):
            s3.download_file(BUCKET, base + suffix, local)
        out[suffix] = local
    return out


# ---------------------------------------------------------------- preprocessing
def eyes_closed_windows(set_path, events_path):
    """Return (n_windows, 19, 3000) eyes-closed windows in volts, plus counts."""
    raw = mne.io.read_raw_eeglab(set_path, preload=True)
    raw.filter(0.5, 75.0)
    raw.notch_filter(60.0)
    # Find noisy or flat electrodes (robust z-score of log standard deviation),
    # rebuild them from their neighbours, then average-reference.
    raw.set_montage(mne.channels.make_standard_montage("GSN-HydroCel-129"))
    egi = [c for c in raw.ch_names if c != "Cz"]       # Cz is the recording reference
    log_sd = np.log(raw.get_data(picks=egi).std(axis=1) + 1e-12)
    mad = np.median(np.abs(log_sd - np.median(log_sd))) + 1e-12
    z = (log_sd - np.median(log_sd)) / (1.4826 * mad)
    raw.info["bads"] = [c for c, zz in zip(egi, z) if abs(zz) > 3]
    if raw.info["bads"]:
        raw.interpolate_bads(reset_bads=True)
    raw.set_eeg_reference("average", projection=False)
    raw.pick(list(EGI_TO_1020))
    raw.rename_channels(EGI_TO_1020)
    raw.reorder_channels(CH_1020)
    raw.resample(SFREQ)
    data = raw.get_data()

    events = pd.read_csv(events_path, sep="\t")
    onsets = events.loc[events["value"] == "instructed_toCloseEyes", "onset"].to_numpy()
    windows, n_total = [], 0
    for onset in onsets:
        start = int((onset + SKIP_SEC) * SFREQ)
        block_end = int((onset + 40) * SFREQ)  # each eyes-closed block is 40 s
        while start + WIN <= min(block_end, data.shape[1]):
            n_total += 1
            w = data[:, start:start + WIN]
            if np.ptp(w, axis=1).max() * 1e6 < PTP_REJECT_UV:
                windows.append(w)
            start += WIN
    windows = np.stack(windows) if windows else np.empty((0, 19, WIN))
    return windows, n_total


# ---------------------------------------------------------------- standard features
def standard_features(windows):
    """Band power, relative power, aperiodic slope/offset, alpha peak. One row per child."""
    from specparam import SpectralModel

    freqs, psd = welch(windows, fs=SFREQ, nperseg=2 * SFREQ, axis=-1)
    psd = psd.mean(axis=0)                      # average over windows -> (19, n_freqs)
    feats = {}
    total = psd[:, (freqs >= 1) & (freqs < 45)].sum(axis=1)
    for band, (lo, hi) in BANDS.items():
        p = psd[:, (freqs >= lo) & (freqs < hi)].sum(axis=1)
        for ch, abs_p, rel_p in zip(CH_1020, p, p / total):
            feats[f"logpow_{band}_{ch}"] = np.log10(abs_p)
            feats[f"relpow_{band}_{ch}"] = rel_p

    def params(fm, kind):
        for name in (kind, kind + "_params"):
            try:
                return fm.get_params(name)
            except Exception:
                continue
        return None

    for ch, spectrum in zip(CH_1020, psd):
        fm = SpectralModel(peak_width_limits=[1, 8], max_n_peaks=6, verbose=False)
        fm.fit(freqs, spectrum, [2, 40])
        offset, exponent = params(fm, "aperiodic")[:2]
        feats[f"aper_offset_{ch}"] = offset
        feats[f"aper_exponent_{ch}"] = exponent

    fm = SpectralModel(peak_width_limits=[1, 8], max_n_peaks=6, verbose=False)
    fm.fit(freqs, psd.mean(axis=0), [2, 40])
    peaks = np.atleast_2d(params(fm, "peak"))
    alpha = peaks[(peaks[:, 0] >= 7) & (peaks[:, 0] <= 14)] if peaks.size else peaks
    feats["alpha_peak_freq"] = alpha[np.argmax(alpha[:, 1]), 0] if len(alpha) else np.nan
    return feats


# ---------------------------------------------------------------- LaBraM
def load_labram(random_weights=False):
    """Pretrained LaBraM, or the same architecture with re-initialized weights."""
    from braindecode.models import Labram

    model = Labram.from_pretrained("braindecode/labram-pretrained").eval()
    if random_weights:
        model = copy.deepcopy(model)
        torch.manual_seed(0)
        for module in model.modules():
            if hasattr(module, "reset_parameters"):
                module.reset_parameters()
        for name, p in model.named_parameters():
            if any(k in name for k in ("position", "cls", "time_embed")):
                torch.nn.init.trunc_normal_(p, std=0.02)
    return model


@torch.no_grad()
def labram_embeddings(model, windows, batch_size=8):
    """Mean token embedding at every layer. Returns (n_layers, 200) for one child."""
    acts = []
    hooks = [b.register_forward_hook(lambda m, i, o: acts.append(o[:, 1:].mean(1)))
             for b in model.blocks]
    per_layer = []
    try:
        x_all = torch.tensor(windows * LABRAM_SCALE, dtype=torch.float32)
        for i in range(0, len(x_all), batch_size):
            acts.clear()
            model(x_all[i:i + batch_size], ch_names=CH_1020, return_features=True)
            per_layer.append(torch.stack(acts, dim=1))   # (batch, n_layers, 200)
    finally:
        for h in hooks:
            h.remove()
    return torch.cat(per_layer).mean(0).numpy()          # average over windows


# ---------------------------------------------------------------- evaluation
ALPHAS = np.logspace(-2, 5, 15)


def _clip(X):
    return np.clip(X, -5, 5)


def scaler():
    """Robust scaling, then clip at 5 IQR-units so one noisy child can't dominate.

    The same treatment is applied to every feature set, so the comparison stays fair.
    """
    return [RobustScaler(), FunctionTransformer(_clip), StandardScaler()]


def ridge():
    return make_pipeline(*scaler(), RidgeCV(alphas=ALPHAS))


def cv_r2(X, y, n_splits=5, n_repeats=5, seed=0):
    """Repeated K-fold R^2 for ridge regression, one child per row."""
    scores = []
    for r in range(n_repeats):
        for tr, te in KFold(n_splits, shuffle=True, random_state=seed + r).split(X):
            m = ridge().fit(X[tr], y[tr])
            resid = y[te] - m.predict(X[te])
            scores.append(1 - (resid ** 2).sum() / ((y[te] - y[tr].mean()) ** 2).sum())
    return float(np.mean(scores)), float(np.std(scores))


def robustness(X, y, n_splits=10, seed=0):
    """Carlson et al. (2024): split children in half, train on each, test on the other.

    P = mean held-out R^2, S = Spearman correlation of the two halves' coefficients,
    R = mean of max(P, 0) * S over random splits.
    """
    rng = np.random.default_rng(seed)
    alpha = ridge().fit(X, y)[-1].alpha_
    P, S = [], []
    for _ in range(n_splits):
        idx = rng.permutation(len(y))
        a, b = idx[: len(y) // 2], idx[len(y) // 2:]
        coefs, perf = [], []
        for tr, te in ((a, b), (b, a)):
            m = make_pipeline(*scaler(), Ridge(alpha=alpha)).fit(X[tr], y[tr])
            resid = y[te] - m.predict(X[te])
            perf.append(1 - (resid ** 2).sum() / ((y[te] - y[te].mean()) ** 2).sum())
            coefs.append(m[-1].coef_)
        P.append(np.mean(perf))
        S.append(spearmanr(coefs[0], coefs[1]).statistic)
    P, S = np.array(P), np.array(S)
    # Clip P at 0 so a model with no skill scores 0 (otherwise a negative P times a
    # negative S would look like positive robustness).
    return {"P": P.mean(), "S": S.mean(), "R": (np.clip(P, 0, None) * S).mean()}

## 3. Look at one child first

Always look at the data before trusting a pipeline. This downloads one child, shows a cleaned eyes-closed window, and plots their spectrum. You should see a clear alpha peak around 8 to 12 Hz in older children, at the back of the head (O1, O2).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import hbn_pipeline as hp

parts = hp.get_participants('R1', DATA_DIR)
parts = parts[parts.RestingState == 'available']
print(len(parts), 'children with resting state in release 1')
print(parts[['participant_id', 'sex', 'age', 'p_factor', 'attention']].head())

sub = parts.participant_id.iloc[0]
paths = hp.download_resting_state('R1', sub, DATA_DIR)
windows, n_total = hp.eyes_closed_windows(paths['eeg.set'], paths['events.tsv'])
print(f'{sub}: kept {len(windows)} of {n_total} eyes-closed windows, shape {windows.shape}')

In [ ]:
from scipy.signal import welch
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
t = np.arange(hp.WIN) / hp.SFREQ
for i, ch in enumerate(hp.CH_1020):
    axes[0].plot(t[:600], windows[0, i, :600] * 1e6 - i * 60, lw=0.6, color='#2a78d6')
axes[0].set_yticks([-i * 60 for i in range(19)], hp.CH_1020, fontsize=7)
axes[0].set_xlabel('seconds'); axes[0].set_title('First 3 s of one window (offset per channel)', loc='left')

freqs, psd = welch(windows, fs=hp.SFREQ, nperseg=2 * hp.SFREQ, axis=-1)
psd = psd.mean(0)
for ch in ['O1', 'O2', 'FZ']:
    axes[1].semilogy(freqs, psd[hp.CH_1020.index(ch)], label=ch)
axes[1].set_xlim(1, 45); axes[1].set_xlabel('Hz'); axes[1].legend(frameon=False)
axes[1].set_title('Power spectrum, eyes closed', loc='left')
plt.tight_layout(); plt.show()

In [ ]:
feats = hp.standard_features(windows)
print(len(feats), 'standard features, e.g.')
for k in ['alpha_peak_freq', 'aper_exponent_O1', 'relpow_alpha_O1', 'logpow_theta_FZ']:
    print(f'  {k}: {feats[k]:.3f}')

model = hp.load_labram()
emb = hp.labram_embeddings(model, windows)
print('LaBraM embedding:', emb.shape, '(12 layers x 200 dimensions, averaged over windows)')

## 4. Process a whole release

This runs every child in release 1 and saves one small `.npz` file per child to `out/`. It skips children it has already done, so if Colab disconnects, just run the cell again and it picks up where it stopped.

Set `MAX_CHILDREN = 20` for a quick test first. Later, add releases `R2`, `R3` and so on: same command, different release name.

In [ ]:
%%writefile run_release.py
"""Process every child with resting-state EEG in one HBN release.

Saves standard features and LaBraM embeddings (pretrained and random weights)
per child, deleting the raw recording afterwards to save disk.
Usage: python run_release.py R1 /path/to/data_dir /path/to/out_dir [max_children]
"""
import os
import sys
import time

import numpy as np
import pandas as pd

import hbn_pipeline as hp

release, data_dir, out_dir = sys.argv[1], sys.argv[2], sys.argv[3]
limit = int(sys.argv[4]) if len(sys.argv) > 4 else None
os.makedirs(out_dir, exist_ok=True)

parts = hp.get_participants(release, data_dir)
parts = parts[parts["RestingState"] == "available"]
if limit:
    parts = parts.head(limit)

pretrained, random_init = hp.load_labram(), hp.load_labram(random_weights=True)
log = []
for i, sub in enumerate(parts["participant_id"]):
    out = os.path.join(out_dir, f"{sub}.npz")
    if os.path.exists(out):
        continue
    t = time.time()
    try:
        paths = hp.download_resting_state(release, sub, data_dir)
        windows, n_total = hp.eyes_closed_windows(paths["eeg.set"], paths["events.tsv"])
        if len(windows) < 2:
            raise ValueError(f"only {len(windows)} clean windows of {n_total}")
        feats = hp.standard_features(windows)
        np.savez(out,
                 feat_names=np.array(list(feats)), feats=np.array(list(feats.values()), float),
                 labram=hp.labram_embeddings(pretrained, windows),
                 labram_random=hp.labram_embeddings(random_init, windows),
                 n_windows=len(windows), n_total=n_total)
        status = f"ok {len(windows)}/{n_total} windows"
    except Exception as e:  # keep going, record the failure
        status = f"FAILED {e!r}"[:200]
    finally:
        for suffix in ("eeg.set", "events.tsv"):
            p = os.path.join(data_dir, f"{sub}_RestingState_{suffix}")
            if os.path.exists(p):
                os.remove(p)
    log.append((sub, status))
    print(f"[{i + 1}/{len(parts)}] {sub} {status} ({time.time() - t:.0f}s)", flush=True)

pd.DataFrame(log, columns=["participant_id", "status"]).to_csv(
    os.path.join(out_dir, f"log_{release}.tsv"), sep="\t", index=False, mode="a")

In [ ]:
RELEASE = 'R3'   # already done: R1, R2, R3. Rerunning only retries children that failed
MAX_CHILDREN = ''   # '' for everyone, or e.g. '20' for a quick test
!python run_release.py {RELEASE} {DATA_DIR} {OUT_DIR} {MAX_CHILDREN} 2>&1 | grep -v -i "warn\|montage\|HF_TOKEN"

## 5. Compare the feature sets

This takes 10 to 20 minutes with three releases. It reports, for each target and feature set:

- **nested_r2** (and **r2_sd**, **r2_median**): cross-validated R², 5 folds repeated 5 times. For LaBraM, the layer is chosen using only the training children in each fold, so the test children never influence it.
- **layer_chosen**: the layer picked most often across the 25 folds.
- **robust_P, prediction_agreement_A, robustness_R**: a version of Carlson et al.'s robustness check. The children are split into thirds, two models are trained on two different thirds, and both predict the last third. A is how closely their predictions agree. R = max(P, 0) × A. Comparing predictions instead of coefficients makes it fair to LaBraM's embedding dimensions, which have no individual meaning.

Then a **paired comparison**: in every test fold, how much better one feature set did than another on the same children, and in how many of the 25 folds it won.

Predictions are capped at the youngest and oldest age in the training set, so one unusual recording can't produce a 32-year-old prediction for a 10-year-old.

In [ ]:
%%writefile analyze.py
"""Compare standard EEG features with LaBraM embeddings on open HBN targets (version 2).

Changes from version 1:
- The LaBraM layer is chosen inside each training set (nested cross-validation),
  so the test children never influence which layer is used.
- Robustness compares the two models' predictions on the same held-out children,
  which works for both interpretable features and arbitrary embedding dimensions.
- Paired comparison: on every test fold, how much better one feature set does than another.

Usage: python analyze.py /path/to/out_dir /path/to/participants.tsv [more participants.tsv ...]
Writes results.csv, layer_results.csv, paired_results.csv and two figures into out_dir.
"""
import glob
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import KFold

import hbn_pipeline as hp

out_dir = sys.argv[1]
parts = pd.concat([pd.read_csv(p, sep="\t") for p in sys.argv[2:]])
parts = parts.drop_duplicates("participant_id").set_index("participant_id")

# ------------------------------------------------------------ load per-child results
files = sorted(glob.glob(os.path.join(out_dir, "sub-*.npz")))
sub_ids = [os.path.basename(f)[:-4] for f in files]
data = [np.load(f) for f in files]
meta = parts.loc[sub_ids]

X_std = np.stack([z["feats"] for z in data])
X_std = np.where(np.isnan(X_std), np.nanmedian(X_std, axis=0), X_std)   # alpha peak can be missing
FEATURE_SETS = {
    "Standard features": X_std[:, None, :],               # one "layer"
    "LaBraM pretrained": np.stack([z["labram"] for z in data]),
    "LaBraM random weights": np.stack([z["labram_random"] for z in data]),
}
TARGETS = {"age": "Age (years)", "attention": "Attention factor", "p_factor": "p-factor"}
print(f"{len(files)} children, {X_std.shape[1]} standard features, "
      f"LaBraM {FEATURE_SETS['LaBraM pretrained'].shape[1]} layers")


# ------------------------------------------------------------ helpers
def r2_score(y_true, y_pred, y_train_mean):
    return 1 - ((y_true - y_pred) ** 2).sum() / ((y_true - y_train_mean) ** 2).sum()


def predict(model, X, y_train):
    """Predict, but never outside the range of the training targets.

    One unusual recording can push a linear model to absurd values (a 10-year-old
    predicted as 32). Capping at the training range is standard in brain-age work
    and is applied identically to every feature set.
    """
    return np.clip(model.predict(X), y_train.min(), y_train.max())


def fit_select(XL, y, seed=0):
    """Pick the best layer by 3-fold CV on this training set only, then fit on it."""
    n_layers = XL.shape[1]
    if n_layers == 1:
        best = 0
    else:
        scores = np.zeros(n_layers)
        for tr, va in KFold(3, shuffle=True, random_state=seed).split(y):
            for layer in range(n_layers):
                m = hp.ridge().fit(XL[tr, layer], y[tr])
                scores[layer] += r2_score(y[va], predict(m, XL[va, layer], y[tr]), y[tr].mean())
        best = int(np.argmax(scores))
    return best, hp.ridge().fit(XL[:, best], y)


def nested_cv(XL, y, n_splits=5, n_repeats=5):
    """Outer repeated K-fold; layer chosen inside each outer training set."""
    scores, layers = [], []
    for r in range(n_repeats):
        for k, (tr, te) in enumerate(KFold(n_splits, shuffle=True, random_state=r).split(y)):
            layer, m = fit_select(XL[tr], y[tr], seed=100 * r + k)
            scores.append(r2_score(y[te], predict(m, XL[te, layer], y[tr]), y[tr].mean()))
            layers.append(layer + 1)
    return np.array(scores), layers


def prediction_robustness(XL, y, n_repeats=10):
    """Split children into thirds. Train one model on each of two thirds, and compare
    their predictions on the remaining third. Rotate so each third is the test set once.

    P = mean held-out R^2 of the two models, A = Spearman agreement of their predictions,
    R = mean of max(P, 0) * A. Unlike coefficient similarity, this does not depend on
    what individual feature dimensions mean, so it is fair to embeddings.
    """
    rng = np.random.default_rng(0)
    P, A = [], []
    for rep in range(n_repeats):
        thirds = np.array_split(rng.permutation(len(y)), 3)
        for t in range(3):
            te, a, b = thirds[t], thirds[(t + 1) % 3], thirds[(t + 2) % 3]
            preds, perf = [], []
            for i, tr in enumerate((a, b)):
                layer, m = fit_select(XL[tr], y[tr], seed=rep * 10 + t * 2 + i)
                p = predict(m, XL[te, layer], y[tr])
                preds.append(p)
                perf.append(r2_score(y[te], p, y[tr].mean()))
            P.append(np.mean(perf))
            A.append(spearmanr(preds[0], preds[1]).statistic)
    P, A = np.array(P), np.nan_to_num(np.array(A))
    return {"P": P.mean(), "A": A.mean(), "R": (np.clip(P, 0, None) * A).mean()}


# ------------------------------------------------------------ per-layer curve (descriptive)
# Every layer scored separately. This is for the figure only: picking the best layer
# from this curve would reuse the test children, which the nested results avoid.
layer_rows = []
for target in TARGETS:
    y = meta[target].to_numpy(float)
    ok = ~np.isnan(y)
    for label in ("LaBraM pretrained", "LaBraM random weights"):
        for layer in range(FEATURE_SETS[label].shape[1]):
            r2, sd = hp.cv_r2(FEATURE_SETS[label][ok, layer], y[ok])
            layer_rows.append({"target": target, "features": label,
                               "layer": layer + 1, "r2": r2, "sd": sd})
layers_df = pd.DataFrame(layer_rows)
layers_df.to_csv(os.path.join(out_dir, "layer_results.csv"), index=False)

# ------------------------------------------------------------ headline comparison
results, fold_scores = [], {}
for target in TARGETS:
    y = meta[target].to_numpy(float)
    ok = ~np.isnan(y)
    for label, XL in FEATURE_SETS.items():
        scores, chosen = nested_cv(XL[ok], y[ok])
        fold_scores[(target, label)] = scores
        rob = prediction_robustness(XL[ok], y[ok])
        common = pd.Series(chosen).value_counts()
        results.append({
            "target": target, "features": label, "n": int(ok.sum()),
            "nested_r2": scores.mean(), "r2_sd": scores.std(), "r2_median": np.median(scores),
            "layer_chosen": "" if label == "Standard features"
                            else f"{common.index[0]} ({common.iloc[0]}/{len(chosen)} folds)",
            "robust_P": rob["P"], "prediction_agreement_A": rob["A"], "robustness_R": rob["R"]})
results = pd.DataFrame(results)
results.to_csv(os.path.join(out_dir, "results.csv"), index=False)

# Paired comparison: the same test children in every fold, so differences are fair.
paired = []
for target in TARGETS:
    for a, b in (("LaBraM pretrained", "Standard features"),
                 ("LaBraM pretrained", "LaBraM random weights"),
                 ("LaBraM random weights", "Standard features")):
        d = fold_scores[(target, a)] - fold_scores[(target, b)]
        paired.append({"target": target, "comparison": f"{a} minus {b}",
                       "mean_r2_difference": d.mean(), "median": np.median(d), "sd": d.std(),
                       "folds_better": f"{(d > 0).sum()}/{len(d)}"})
paired = pd.DataFrame(paired)
paired.to_csv(os.path.join(out_dir, "paired_results.csv"), index=False)

pd.set_option("display.width", 200)
print("\nNested cross-validation and prediction robustness")
print(results.round(3).to_string(index=False))
print("\nPaired differences in R² (same test children)")
print(paired.round(3).to_string(index=False))

# ------------------------------------------------------------ figures
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1a1a19", "#6b6a63", "#e6e5e0"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False})

# Figure 1: age R^2 by layer, with the nested standard-feature level as a reference line
fig, ax = plt.subplots(figsize=(7, 4))
age = layers_df.query("target == 'age'")
for label, color in (("LaBraM pretrained", BLUE), ("LaBraM random weights", ORANGE)):
    d = age.query("features == @label")
    ax.plot(d.layer, d.r2, color=color, lw=2, marker="o", ms=5, label=label)
std_age = results.query("target == 'age' and features == 'Standard features'").nested_r2.iloc[0]
ax.axhline(std_age, color=INK, lw=1.2, ls="--", label="Standard features")
ax.set_xlabel("LaBraM layer")
ax.set_ylabel("Cross-validated R², predicting age")
ax.set_xticks(range(1, 13))
ax.grid(axis="y", color=GRID, lw=0.8)
ax.legend(frameon=False, loc="lower left")
ax.set_title(f"Predicting age from resting EEG, by LaBraM layer ({len(files)} HBN children)",
             loc="left", color=INK)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, "fig1_age_by_layer.png"), dpi=200)

# Figure 2: nested R^2 (with fold-to-fold spread) and prediction robustness, by target
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = list(FEATURE_SETS)
colors = [INK, BLUE, ORANGE]
x = np.arange(len(TARGETS))
w = 0.26
for ax, metric, err, title in ((axes[0], "nested_r2", "r2_sd", "Nested cross-validated R²"),
                               (axes[1], "robustness_R", None, "Prediction robustness R")):
    for i, label in enumerate(labels):
        sub = results.set_index(["target", "features"])
        vals = [sub.loc[(t, label), metric] for t in TARGETS]
        errs = [sub.loc[(t, label), err] for t in TARGETS] if err else None
        ax.bar(x + (i - 1) * w, vals, w - 0.02, color=colors[i], label=label,
               yerr=errs, error_kw={"ecolor": MUTED, "lw": 1, "capsize": 2})
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_xticks(x, list(TARGETS.values()))
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_title(title, loc="left", color=INK)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, "fig2_comparison.png"), dpi=200)
print("\nsaved figures")

In [ ]:
import glob
!python analyze.py {OUT_DIR} {' '.join(glob.glob(os.path.join(DATA_DIR, 'participants_*.tsv')))}

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(OUT_DIR, 'fig1_age_by_layer.png'), width=700))
display(Image(os.path.join(OUT_DIR, 'fig2_comparison.png'), width=900))

## What to look at

1. **The paired table is the main result.** "LaBraM pretrained minus Standard features" tells you whether the foundation model knows more about these children than the standard features. Look at the median difference and how many of 25 folds it wins, not just the mean.
2. **Pretrained minus random weights** tells you whether pretraining adds anything beyond the model's structure.
3. **Which layer gets chosen.** If the same layer wins most folds, that's the layer to study with sparse autoencoders later.
4. **Robustness R.** Higher means models trained on different children make the same predictions, the property Carlson's paper argues matters more than raw accuracy.
5. **Attention and p-factor near zero** means resting EEG doesn't predict those questionnaire scores, for any feature set.